# 02g — Devices

Procesa `data/data_raw/devices.csv` para generar features de plataforma
y recencia a nivel de usuario.

**CSV de entrada:** 144 MB, 1.129.784 filas, 6 columnas:
`_id`, `user_id`, `platform`, `device_model`, `updated_at`, `created_at`.

**Hallazgo clave:** ~9 devices/user de media. Probablemente cada
actualización crea una nueva fila (logs de sesión) o cada reinstalación
registra un device nuevo. Por eso `device_records_total` y
`device_unique_models` miden cosas distintas.

**Política point-in-time:** descartar filas con `created_at > REFERENCE_DATE`
O `updated_at > REFERENCE_DATE`. Si el registro no existía en el cutoff,
no debe contribuir a ninguna feature.

**Outputs:**
- `data/data_qc/features_devices.parquet` (126.253 × 11)
- `informes/fase1_cleaning/devices/execution_report.md`
- HTML + sección enriquecida

In [1]:
# [SETUP] Imports y dependencias
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc
import time
from pathlib import Path
from datetime import datetime

plt.style.use('default')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

In [2]:
# [SETUP] Paths, constantes y helpers
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase1_cleaning' else Path.cwd()
DATA_RAW = PROJECT_ROOT / 'data' / 'data_raw'
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase1_cleaning' / 'devices'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CSV_INPUT = DATA_RAW / 'devices.csv'
SAMPLE_IDS_PATH = DATA_QC / 'sample_user_ids_cutoff.parquet'
PARQUET_FEATURES = DATA_QC / 'features_devices_cutoff.parquet'

def clean_user_id(uid):
    uid = str(uid)
    if uid.startswith('ObjectId(') and uid.endswith(')'):
        return uid[9:-1].replace("'", "")
    return uid

steps_log = []
NOTEBOOK_START = time.time()

def log_step(tag, message):
    ts = datetime.now().strftime('%H:%M:%S')
    entry = f"[{tag}] {ts} — {message}"
    steps_log.append(entry)
    print(entry)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CSV_INPUT existe: {CSV_INPUT.exists()}")

PROJECT_ROOT: /Users/jezquerro/Documents/tfg
CSV_INPUT existe: True


In [3]:
# [SETUP] Carga sample_user_ids, REFERENCE_DATE y sentinel
sample_ids_df = pd.read_parquet(SAMPLE_IDS_PATH)
sample_ids_df['user_id'] = sample_ids_df['user_id'].apply(clean_user_id)
sample_user_ids = set(sample_ids_df['user_id'])
N_SAMPLE = len(sample_user_ids)

sample_example = list(sample_user_ids)[0]
assert len(sample_example) == 24 and not sample_example.startswith('ObjectId'), \
    f"ERROR: user_id no es hex limpio: '{sample_example}'"

users_clean = pd.read_parquet(DATA_QC / 'users_clean_cutoff.parquet', columns=['last_login_dt'])
REFERENCE_DATE = users_clean['last_login_dt'].max()
CUTOFF_DATE = REFERENCE_DATE - pd.Timedelta(days=120)
del users_clean
gc.collect()

REFERENCE_DATE_UNIX = int(REFERENCE_DATE.timestamp())
CUTOFF_DATE_UNIX = int(CUTOFF_DATE.timestamp())
SENTINEL_DAYS = 9999

print(f"Usuarios en sample: {N_SAMPLE:,}")
print(f"REFERENCE_DATE: {REFERENCE_DATE}")
print(f"REFERENCE_DATE_UNIX: {REFERENCE_DATE_UNIX}")
print(f"SENTINEL_DAYS: {SENTINEL_DAYS}")

log_step('SETUP', f'sample_user_ids: {N_SAMPLE:,} usuarios')
log_step('SETUP', f'REFERENCE_DATE_UNIX: {REFERENCE_DATE_UNIX}')

Usuarios en sample: 25,200
REFERENCE_DATE: 2026-04-04 18:23:13
REFERENCE_DATE_UNIX: 1775326993
SENTINEL_DAYS: 9999
[SETUP] 13:13:23 — sample_user_ids: 25,200 usuarios
[SETUP] 13:13:23 — REFERENCE_DATE_UNIX: 1775326993


## 1. Carga y exploración del CSV

In [4]:
# [EXEC] Cargar devices.csv completo
t0 = time.time()
df_raw = pd.read_csv(CSV_INPUT, low_memory=False)
load_time = time.time() - t0

print(f"Shape: {df_raw.shape}")
print(f"Tiempo: {load_time:.1f}s")
print(f"Memoria: {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")
log_step('EXEC', f'Carga CSV: shape={df_raw.shape}, tiempo={load_time:.1f}s')

Shape: (1129784, 6)
Tiempo: 1.9s
Memoria: 199.2 MB
[EXEC] 13:13:25 — Carga CSV: shape=(1129784, 6), tiempo=1.9s


In [5]:
# [ANALYSIS] Exploración devices — sobre CSV COMPLETO
print("TIPOS DE DATOS — devices raw")
print("=" * 80)
for col in df_raw.columns:
    dtype = df_raw[col].dtype
    n_nulls = df_raw[col].isnull().sum()
    pct_nulls = n_nulls / len(df_raw) * 100
    n_unique = df_raw[col].nunique()
    example = df_raw[col].dropna().iloc[0] if n_nulls < len(df_raw) else 'TODO NULO'
    print(f"  {col:<20} dtype={str(dtype):<10} nulls={n_nulls:>8,} ({pct_nulls:5.1f}%)  "
          f"unique={n_unique:>8,}  ej='{str(example)[:30]}'")

# Nulos por columna → CSV
nulls_raw = df_raw.isnull().sum().to_frame(name='n_nulls')
nulls_raw['pct_nulls'] = (nulls_raw['n_nulls'] / len(df_raw) * 100).round(2)
nulls_raw = nulls_raw.sort_values('pct_nulls', ascending=False)
nulls_raw.to_csv(REPORT_DIR / 'nulos_por_columna_raw.csv')

# % user_id NaN sobre CSV COMPLETO
pct_uid_nan = df_raw['user_id'].isna().mean() * 100
print(f"\n% user_id NaN (CSV completo): {pct_uid_nan:.3f}%")

# Distribución de platform
print("\n=== platform ===")
print(df_raw['platform'].value_counts(dropna=False))

# Distribución de device_model (top 10)
print("\n=== device_model (top 10) ===")
print(df_raw['device_model'].value_counts(dropna=False).head(10))

# Rango temporal
print("\n=== Rango temporal (created_at) ===")
ca_sample = pd.to_datetime(df_raw['created_at'].head(5).tolist(), utc=True, errors='coerce')
print(f"  ejemplos ca: {ca_sample.tolist()[:3]}")

# Parseo mínimo para reportar rango
ca_min = pd.to_datetime(df_raw['created_at'], utc=True, errors='coerce').min()
ca_max = pd.to_datetime(df_raw['created_at'], utc=True, errors='coerce').max()
print(f"  min created_at: {ca_min}")
print(f"  max created_at: {ca_max}")

log_step('ANALYSIS', f'Devices raw: {pct_uid_nan:.3f}% NaN, '
         f'{df_raw["platform"].nunique()} plataformas, '
         f'{df_raw["device_model"].nunique()} modelos')

TIPOS DE DATOS — devices raw
  _id                  dtype=str        nulls=       0 (  0.0%)  unique=1,129,784  ej='ObjectId(6194fc0aa1bb376e0b35f'
  user_id              dtype=str        nulls=       0 (  0.0%)  unique=1,076,057  ej='6118c8d3b7e3351ef15b53f2'
  platform             dtype=str        nulls=       0 (  0.0%)  unique=       2  ej='android'


  device_model         dtype=str        nulls=       3 (  0.0%)  unique=  14,333  ej='HUAWEI BMH-AN10'
  updated_at           dtype=str        nulls=       0 (  0.0%)  unique=1,129,703  ej='2021-11-17T12:56:42.669Z'
  created_at           dtype=str        nulls=       0 (  0.0%)  unique=1,129,703  ej='2021-11-17T12:56:42.669Z'

% user_id NaN (CSV completo): 0.000%

=== platform ===
platform
android    1009463
ios         120321
Name: count, dtype: int64

=== device_model (top 10) ===
device_model
iPhone12,1                 13515
iPhone14,5                 11007
Xiaomi M2006C3LG           10023
Xiaomi M2102J20SG           6432
iPhone13,2                  6230
Xiaomi Redmi Note 8 Pro     5968
Xiaomi M2004J19C            5820
realme RMX3834              5187
iPhone14,7                  5024
Xiaomi M2003J15SC           4949
Name: count, dtype: int64

=== Rango temporal (created_at) ===
  ejemplos ca: [Timestamp('2021-11-17 12:56:42.669000+0000', tz='UTC'), Timestamp('2021-11-17 15:25:40.94

  min created_at: 2021-11-17 12:56:42.669000+00:00
  max created_at: 2026-04-04 18:15:07.155000+00:00
[ANALYSIS] 13:13:26 — Devices raw: 0.000% NaN, 2 plataformas, 14333 modelos


## 2. Filtrado point-in-time

Cuatro filtros secuenciales:
1. Eliminar filas con `user_id` NaN.
2. Eliminar filas con `created_at > REFERENCE_DATE` (no existían en el cutoff).
3. Eliminar filas con `updated_at > REFERENCE_DATE` (actualización posterior no visible).
4. Filtrar por `sample_user_ids`.

In [6]:
# [EXEC] Parsear fechas y aplicar filtros
t0 = time.time()
n_before = len(df_raw)

# 1. Filtro 1: user_id NaN
df = df_raw.dropna(subset=['user_id']).copy()
n_no_nan = len(df)

# 2. Normalizar user_id (salvaguarda)
df['user_id'] = df['user_id'].apply(clean_user_id)

# 3. Parser de timestamps ISO8601 → unix seconds (robusto, independiente de la unidad interna)
# En pandas 3.x, pd.to_datetime(..., utc=True) devuelve datetime64[us, UTC] (microsegundos),
# no [ns]. La resta contra epoch con Timedelta('1s') es independiente de la unidad.
EPOCH_UTC = pd.Timestamp('1970-01-01', tz='UTC')

created_dt = pd.to_datetime(df['created_at'], utc=True, errors='coerce')
updated_dt = pd.to_datetime(df['updated_at'], utc=True, errors='coerce')

df['created_at_unix'] = (
    (created_dt - EPOCH_UTC) // pd.Timedelta('1s')
).astype('Int64')  # Int64 nullable: NaT → NA automáticamente
df['updated_at_unix'] = (
    (updated_dt - EPOCH_UTC) // pd.Timedelta('1s')
).astype('Int64')

# Validación inmediata del rango unix (sobre valores válidos)
valid_created = df['created_at_unix'].dropna()
valid_updated = df['updated_at_unix'].dropna()
assert valid_created.min() > 1_000_000_000, (
    f"created_at_unix corrupto: min={valid_created.min()}. "
    "Debería ser ~1.5e9-1.8e9 (años 2017-2026)."
)
assert valid_created.max() < 2_000_000_000, (
    f"created_at_unix corrupto: max={valid_created.max()}. "
    "Debería ser <2e9 (antes de 2033)."
)
assert valid_updated.min() > 1_000_000_000, (
    f"updated_at_unix corrupto: min={valid_updated.min()}."
)
assert valid_updated.max() < 2_000_000_000, (
    f"updated_at_unix corrupto: max={valid_updated.max()}."
)
print(f"  created_at_unix rango: [{int(valid_created.min())}, {int(valid_created.max())}]")
print(f"  updated_at_unix rango: [{int(valid_updated.min())}, {int(valid_updated.max())}]")

# 4. Filtros point-in-time
n_step = len(df)
df = df[df['created_at_unix'].notna() & (df['created_at_unix'] <= CUTOFF_DATE_UNIX)].copy()
n_created_filter = n_step - len(df)
n_created_ok = len(df)

n_step = len(df)
df = df[df['updated_at_unix'].notna() & (df['updated_at_unix'] <= CUTOFF_DATE_UNIX)].copy()
n_updated_filter = n_step - len(df)
n_updated_ok = len(df)

# 5. Filtro 4: sample_user_ids
df = df[df['user_id'].isin(sample_user_ids)].copy()
n_in_sample = len(df)

# Conservar también con los nombres heredados por compatibilidad con el resto del notebook
df['created_unix'] = df['created_at_unix']
df['updated_unix'] = df['updated_at_unix']

print(f"\nFilas iniciales:                 {n_before:>12,}")
print(f"Tras eliminar user_id NaN:       {n_no_nan:>12,}  ({n_no_nan/n_before*100:.2f}%)")
print(f"Tras filtro created<=REF:        {n_created_ok:>12,}  ({n_created_ok/n_before*100:.2f}%)  [-{n_created_filter:,}]")
print(f"Tras filtro updated<=REF:        {n_updated_ok:>12,}  ({n_updated_ok/n_before*100:.2f}%)  [-{n_updated_filter:,}]")
print(f"Tras filtro sample_user_ids:     {n_in_sample:>12,}  ({n_in_sample/n_before*100:.2f}%)")

n_users_with_devices = df['user_id'].nunique()
print(f"\nUsuarios con devices: {n_users_with_devices:,} ({n_users_with_devices/N_SAMPLE*100:.2f}%)")
print(f"Usuarios sin devices: {N_SAMPLE - n_users_with_devices:,} ({(N_SAMPLE - n_users_with_devices)/N_SAMPLE*100:.2f}%)")
print(f"Tiempo filtrado: {time.time()-t0:.1f}s")

log_step('EXEC', f'Filtro created_at <= REF: -{n_created_filter} filas')
log_step('EXEC', f'Filtro updated_at <= REF: -{n_updated_filter} filas')
log_step('EXEC', f'Filtrado: {n_before:,} → {n_in_sample:,}')
log_step('EXEC', f'Usuarios con devices: {n_users_with_devices:,}')

del df_raw
gc.collect()

  created_at_unix rango: [1637153802, 1775326507]
  updated_at_unix rango: [1637153802, 1775326507]

Filas iniciales:                    1,129,784
Tras eliminar user_id NaN:          1,129,784  (100.00%)
Tras filtro created<=REF:             720,229  (63.75%)  [-409,555]
Tras filtro updated<=REF:             714,114  (63.21%)  [-6,115]
Tras filtro sample_user_ids:           23,534  (2.08%)

Usuarios con devices: 20,708 (82.17%)
Usuarios sin devices: 4,492 (17.83%)
Tiempo filtrado: 1.2s
[EXEC] 13:13:27 — Filtro created_at <= REF: -409555 filas
[EXEC] 13:13:27 — Filtro updated_at <= REF: -6115 filas
[EXEC] 13:13:27 — Filtrado: 1,129,784 → 23,534
[EXEC] 13:13:27 — Usuarios con devices: 20,708


0

In [7]:
# [ANALYSIS] Estadísticas post-filtro
print(f"Filas filtradas: {len(df):,}")
print(f"Usuarios únicos: {df['user_id'].nunique():,}")
print(f"Memoria: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

# Distribución devices/user
devices_per_user = df.groupby('user_id').size()
print(f"\nDevices por usuario (post-filtro):")
print(f"  Mediana: {devices_per_user.median():.0f}")
print(f"  Media:   {devices_per_user.mean():.2f}")
print(f"  Min:     {devices_per_user.min()}")
print(f"  Max:     {devices_per_user.max()}")
print(f"  P90:     {devices_per_user.quantile(0.9):.0f}")
print(f"  P99:     {devices_per_user.quantile(0.99):.0f}")

# Distribución platform post-filtro
print(f"\n=== platform post-filtro ===")
print(df['platform'].value_counts(dropna=False))

log_step('ANALYSIS', f'Post-filtro: devices/user mediana={devices_per_user.median():.0f}, '
         f'max={devices_per_user.max()}')

Filas filtradas: 23,534
Usuarios únicos: 20,708
Memoria: 5.2 MB

Devices por usuario (post-filtro):
  Mediana: 1
  Media:   1.14
  Min:     1
  Max:     92
  P90:     1
  P99:     3

=== platform post-filtro ===
platform
android    19154
ios         4380
Name: count, dtype: int64
[ANALYSIS] 13:13:28 — Post-filtro: devices/user mediana=1, max=92


## 3. Agregación por usuario — 10 features en 3 grupos

- **A:** Volumen de devices (3)
- **B:** Plataforma (4)
- **C:** Temporales point-in-time (3)

In [8]:
# [EXEC] Grupo A — Volumen de devices (3 features)
t0 = time.time()

if len(df) > 0:
    grp_a = df.groupby('user_id').agg(
        device_records_total=('_id', 'size'),
        device_unique_models=('device_model', 'nunique'),
        device_platform_count=('platform', 'nunique'),
    )
else:
    grp_a = pd.DataFrame(columns=['device_records_total', 'device_unique_models', 'device_platform_count'])

print(f"Grupo A: {len(grp_a):,} filas, {grp_a.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo A (volumen): {grp_a.shape[1]} features')

Grupo A: 20,708 filas, 3 features, 0.0s
[EXEC] 13:13:28 — Grupo A (volumen): 3 features


In [9]:
# [EXEC] Grupo B — Plataforma (4 features)
t0 = time.time()

if len(df) > 0:
    # Flags android / ios
    users_android = set(df[df['platform'] == 'android']['user_id'].unique())
    users_ios = set(df[df['platform'] == 'ios']['user_id'].unique())

    # DataFrame del grupo B (index = usuarios con devices)
    users_with_dev = df['user_id'].unique()
    grp_b = pd.DataFrame(index=pd.Index(users_with_dev, name='user_id'))
    grp_b['device_has_android'] = grp_b.index.isin(users_android).astype(int)
    grp_b['device_has_ios'] = grp_b.index.isin(users_ios).astype(int)
    grp_b['device_is_multi_platform'] = (
        (grp_b['device_has_android'] == 1) & (grp_b['device_has_ios'] == 1)
    ).astype(int)

    # device_primary_platform: moda de platform por user_id
    # Usar value_counts y quedarse con la plataforma más frecuente
    platform_counts = df.groupby(['user_id', 'platform']).size().reset_index(name='n')
    # Sort: por n descendente, luego platform ascendente (para desempatar alfabéticamente)
    platform_counts = platform_counts.sort_values(
        ['user_id', 'n', 'platform'], ascending=[True, False, True]
    )
    primary = platform_counts.drop_duplicates(subset='user_id', keep='first').set_index('user_id')['platform']
    grp_b['device_primary_platform'] = primary
else:
    grp_b = pd.DataFrame(columns=[
        'device_has_android', 'device_has_ios',
        'device_is_multi_platform', 'device_primary_platform',
    ])

print(f"Grupo B: {len(grp_b):,} filas, {grp_b.shape[1]} features, {time.time()-t0:.1f}s")
if len(grp_b) > 0:
    print(f"  device_has_android: {grp_b['device_has_android'].sum():,}")
    print(f"  device_has_ios:     {grp_b['device_has_ios'].sum():,}")
    print(f"  device_is_multi_platform: {grp_b['device_is_multi_platform'].sum():,}")
    print(f"  device_primary_platform value_counts:")
    print(grp_b['device_primary_platform'].value_counts().to_string())
log_step('EXEC', f'Grupo B (plataforma): {grp_b.shape[1]} features')

Grupo B: 20,708 filas, 4 features, 0.1s
  device_has_android: 17,101
  device_has_ios:     3,625
  device_is_multi_platform: 18
  device_primary_platform value_counts:
device_primary_platform
android    17095
ios         3613
[EXEC] 13:13:28 — Grupo B (plataforma): 4 features


In [10]:
# [EXEC] Grupo C — Temporales point-in-time (3 features)
t0 = time.time()

if len(df) > 0:
    grp_c = df.groupby('user_id').agg(
        first_seen_unix=('created_unix', 'min'),
        last_active_unix=('updated_unix', 'max'),
    )
    # Convertir a datetime explícitamente con unit='s' (CRÍTICO: sin esto sale 1970)
    grp_c['device_first_seen_dt'] = pd.to_datetime(
        grp_c['first_seen_unix'], unit='s', errors='coerce'
    )
    grp_c['device_last_active_dt'] = pd.to_datetime(
        grp_c['last_active_unix'], unit='s', errors='coerce'
    )

    # device_days_since_last_active: calcular sobre last_active_unix
    days_since = ((CUTOFF_DATE_UNIX - grp_c['last_active_unix']) / 86400).clip(lower=0)
    grp_c['device_days_since_last_active'] = days_since.astype(int)

    # Quedarnos solo con las 3 features finales
    grp_c = grp_c[['device_first_seen_dt', 'device_last_active_dt', 'device_days_since_last_active']]

    # Validación de rango temporal absoluto (post-fix)
    valid_dates = grp_c['device_first_seen_dt'].dropna()
    if len(valid_dates) > 0:
        assert valid_dates.min().year >= 2018, (
            f"device_first_seen_dt fuera de rango: min={valid_dates.min()}"
        )
        assert valid_dates.max().year <= 2027, (
            f"device_first_seen_dt fuera de rango: max={valid_dates.max()}"
        )
        print(f"  device_first_seen_dt rango: [{valid_dates.min()}, {valid_dates.max()}]")

    valid_dates_la = grp_c['device_last_active_dt'].dropna()
    if len(valid_dates_la) > 0:
        assert valid_dates_la.min().year >= 2018, (
            f"device_last_active_dt fuera de rango: min={valid_dates_la.min()}"
        )
        assert valid_dates_la.max().year <= 2027, (
            f"device_last_active_dt fuera de rango: max={valid_dates_la.max()}"
        )
        print(f"  device_last_active_dt rango: [{valid_dates_la.min()}, {valid_dates_la.max()}]")
else:
    grp_c = pd.DataFrame(columns=[
        'device_first_seen_dt', 'device_last_active_dt', 'device_days_since_last_active',
    ])

print(f"Grupo C: {len(grp_c):,} filas, {grp_c.shape[1]} features, {time.time()-t0:.1f}s")
if len(grp_c) > 0:
    print(f"  device_days_since_last_active describe:")
    print(grp_c['device_days_since_last_active'].describe().to_string())
log_step('EXEC', f'Grupo C (temporales): {grp_c.shape[1]} features')

  device_first_seen_dt rango: [2021-11-17 15:25:40, 2025-12-05 18:22:43]
  device_last_active_dt rango: [2021-11-24 12:58:31, 2025-12-05 18:22:43]
Grupo C: 20,708 filas, 3 features, 0.0s
  device_days_since_last_active describe:
count    20708.000000
mean       149.546552
std        224.258867
min          0.000000
25%          9.000000
50%         53.000000
75%        202.000000
max       1472.000000
[EXEC] 13:13:28 — Grupo C (temporales): 3 features


In [11]:
# [EXEC] Combinar grupos + reindex con sample_user_ids
t0 = time.time()

features = pd.concat([grp_a, grp_b, grp_c], axis=1)
features = features.reindex(list(sample_user_ids))
features.index.name = 'user_id'
features = features.reset_index()

# Fills por columna según tipo
int_flag_cols = [
    'device_records_total', 'device_unique_models', 'device_platform_count',
    'device_has_android', 'device_has_ios', 'device_is_multi_platform',
]
for col in int_flag_cols:
    features[col] = features[col].fillna(0).astype(int)

# device_primary_platform: 'none' si no tiene devices
features['device_primary_platform'] = features['device_primary_platform'].fillna('none').astype(str)

# device_days_since_last_active: sentinel 9999 si NaN
features['device_days_since_last_active'] = (
    features['device_days_since_last_active'].fillna(SENTINEL_DAYS).astype(int)
)

# Datetimes se mantienen con NaT (no fillna)

# Verificación
assert len(features) == N_SAMPLE, f"ERROR: {len(features)} != {N_SAMPLE}"
assert features.shape[1] == 11, f"ERROR: {features.shape[1]} cols, esperado 11"

print(f"Features finales: {features.shape}")
print(f"Verificación: {len(features)} filas == {N_SAMPLE}, {features.shape[1]} cols == 11")
print(f"\nColumnas: {list(features.columns)}")
log_step('EXEC', f'Features combinadas: {features.shape[0]:,} × {features.shape[1]} cols')

Features finales: (25200, 11)
Verificación: 25200 filas == 25200, 11 cols == 11

Columnas: ['user_id', 'device_records_total', 'device_unique_models', 'device_platform_count', 'device_has_android', 'device_has_ios', 'device_is_multi_platform', 'device_primary_platform', 'device_first_seen_dt', 'device_last_active_dt', 'device_days_since_last_active']
[EXEC] 13:13:28 — Features combinadas: 25,200 × 11 cols


## 4. Sanity checks — obligatorios antes de guardar

In [12]:
# [ANALYSIS] Sanity checks lógicos (bloquea guardado si fallan)

errors = []

def _check(condition, msg):
    if condition:
        print(f"  {msg}")
    else:
        print(f"  {msg}")
        errors.append(msg)

# 1. Shape
_check(len(features) == N_SAMPLE, f"Shape = 126,253 (got {len(features)})")

# 2. user_id único
_check(features['user_id'].nunique() == N_SAMPLE,
       f"user_id único = 126,253 (got {features['user_id'].nunique()})")

# 3. Counts no-negativos
_check(features['device_records_total'].min() >= 0, "device_records_total >= 0")
_check(features['device_unique_models'].min() >= 0, "device_unique_models >= 0")
_check(features['device_platform_count'].min() >= 0, "device_platform_count >= 0")

# 4. Usuarios con registros tienen platform_count >= 1
mask = features['device_records_total'] > 0
if mask.sum() > 0:
    _check(features.loc[mask, 'device_platform_count'].min() >= 1,
           "records>0 → platform_count >= 1")

# 5. is_multi_platform=1 → has_android=1 AND has_ios=1
mask = features['device_is_multi_platform'] == 1
if mask.sum() > 0:
    both_flags = (features.loc[mask, 'device_has_android'] == 1) & (features.loc[mask, 'device_has_ios'] == 1)
    _check(both_flags.all(), "is_multi_platform=1 → has_android=1 AND has_ios=1")
else:
    print("  ℹNingún usuario multi-plataforma (saltando check)")

# 6. is_multi_platform=1 → platform_count >= 2
mask = features['device_is_multi_platform'] == 1
if mask.sum() > 0:
    _check(features.loc[mask, 'device_platform_count'].min() >= 2,
           "is_multi_platform=1 → platform_count >= 2")

# 7. Usuarios con devices tienen al menos una plataforma (android OR ios) — esperado para la mayoría
mask = features['device_records_total'] > 0
if mask.sum() > 0:
    has_any = (features.loc[mask, 'device_has_android'] == 1) | (features.loc[mask, 'device_has_ios'] == 1)
    # No es estrictamente obligatorio si hay otras plataformas ('windows', 'web'). Reportar.
    n_no_mainstream = (~has_any).sum()
    if n_no_mainstream > 0:
        print(f"  ℹ{n_no_mainstream} usuarios con devices pero sin android ni ios (otras plataformas)")
    # Solo failure si el 100% de usuarios-con-devices no tienen NI android NI ios (muy improbable)
    _check(has_any.sum() > 0, "Al menos un usuario con devices tiene android o ios")

# 8. first_seen_dt <= last_active_dt cuando no son NaT
mask = features['device_first_seen_dt'].notna() & features['device_last_active_dt'].notna()
if mask.sum() > 0:
    _check((features.loc[mask, 'device_first_seen_dt'] <=
            features.loc[mask, 'device_last_active_dt']).all(),
           "first_seen_dt <= last_active_dt (cuando ambos no-NaT)")

# 9. days_since_last_active == 9999 ↔ last_active_dt is NaT
nat_mask = features['device_last_active_dt'].isna()
sentinel_mask = features['device_days_since_last_active'] == SENTINEL_DAYS
_check((nat_mask == sentinel_mask).all(),
       "days_since==9999 ↔ last_active_dt is NaT")

print()
if errors:
    raise AssertionError(f"{len(errors)} sanity checks fallaron — NO guardar parquet")
else:
    print(f"TODOS los sanity checks pasaron (9/9)")
log_step('ANALYSIS', f'Sanity checks: {len(errors)} errores')

  Shape = 126,253 (got 25200)
  user_id único = 126,253 (got 25200)
  device_records_total >= 0
  device_unique_models >= 0
  device_platform_count >= 0
  records>0 → platform_count >= 1
  is_multi_platform=1 → has_android=1 AND has_ios=1
  is_multi_platform=1 → platform_count >= 2
  Al menos un usuario con devices tiene android o ios
  first_seen_dt <= last_active_dt (cuando ambos no-NaT)
  days_since==9999 ↔ last_active_dt is NaT

TODOS los sanity checks pasaron (9/9)
[ANALYSIS] 13:13:28 — Sanity checks: 0 errores


## 5. Validación de features

In [13]:
# [ANALYSIS] Tipos de datos finales + zeros vs nonzeros
print("TIPOS DE DATOS — FEATURES FINALES")
print("=" * 80)
for col in features.columns:
    dtype = features[col].dtype
    n_nulls = features[col].isnull().sum()
    if pd.api.types.is_numeric_dtype(features[col]):
        n_zeros = (features[col] == 0).sum()
        n_nonzero = len(features) - n_nulls - n_zeros
        print(f"  {col:<40} dtype={str(dtype):<15} nulls={n_nulls:>8,}  "
              f"zeros={n_zeros:>8,}  nonzero={n_nonzero:>8,}")
    else:
        print(f"  {col:<40} dtype={str(dtype):<15} nulls={n_nulls:>8,}")

# value_counts para primary_platform
print("\n=== device_primary_platform ===")
print(features['device_primary_platform'].value_counts(dropna=False))

TIPOS DE DATOS — FEATURES FINALES
  user_id                                  dtype=str             nulls=       0
  device_records_total                     dtype=int64           nulls=       0  zeros=   4,492  nonzero=  20,708
  device_unique_models                     dtype=int64           nulls=       0  zeros=   4,492  nonzero=  20,708
  device_platform_count                    dtype=int64           nulls=       0  zeros=   4,492  nonzero=  20,708
  device_has_android                       dtype=int64           nulls=       0  zeros=   8,099  nonzero=  17,101
  device_has_ios                           dtype=int64           nulls=       0  zeros=  21,575  nonzero=   3,625
  device_is_multi_platform                 dtype=int64           nulls=       0  zeros=  25,182  nonzero=      18
  device_primary_platform                  dtype=str             nulls=       0
  device_first_seen_dt                     dtype=datetime64[s]   nulls=   4,492
  device_last_active_dt                   

In [14]:
# [ANALYSIS] Nulos en features finales
nulls_f = features.isnull().sum().to_frame(name='n_nulls')
nulls_f['pct_nulls'] = (nulls_f['n_nulls'] / len(features) * 100).round(2)
nulls_f = nulls_f[nulls_f['n_nulls'] > 0].sort_values('pct_nulls', ascending=False)

if len(nulls_f) > 0:
    print("Columnas con nulos (esperado solo en datetimes):")
    print(nulls_f)
else:
    print("No hay nulos")

nulls_f.to_csv(REPORT_DIR / 'nulos_features_final.csv')
log_step('ANALYSIS', f'Nulos: {len(nulls_f)} columnas con nulos')

Columnas con nulos (esperado solo en datetimes):
                       n_nulls  pct_nulls
device_first_seen_dt      4492      17.83
device_last_active_dt     4492      17.83
[ANALYSIS] 13:13:28 — Nulos: 2 columnas con nulos


In [15]:
# [ANALYSIS] Estadísticas descriptivas
numeric_features = features.select_dtypes(include=[np.number])
desc = numeric_features.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).T.round(2)
print(desc)
desc.to_csv(REPORT_DIR / 'features_describe.csv')
log_step('ANALYSIS', 'Estadísticas descriptivas guardadas')

                                 count     mean      std  min   25%   50%    75%     90%     99%     max
device_records_total           25200.0     0.93     1.07  0.0   1.0   1.0    1.0     1.0     3.0    92.0
device_unique_models           25200.0     0.89     0.76  0.0   1.0   1.0    1.0     1.0     2.0    77.0
device_platform_count          25200.0     0.82     0.38  0.0   1.0   1.0    1.0     1.0     1.0     2.0
device_has_android             25200.0     0.68     0.47  0.0   0.0   1.0    1.0     1.0     1.0     1.0
device_has_ios                 25200.0     0.14     0.35  0.0   0.0   0.0    0.0     1.0     1.0     1.0
device_is_multi_platform       25200.0     0.00     0.03  0.0   0.0   0.0    0.0     0.0     0.0     1.0
device_days_since_last_active  25200.0  1905.25  3775.20  0.0  14.0  98.0  450.0  9999.0  9999.0  9999.0
[ANALYSIS] 13:13:28 — Estadísticas descriptivas guardadas


In [16]:
# [ANALYSIS] Histogramas de features clave
key_features = [
    'device_records_total',
    'device_unique_models',
    'device_platform_count',
    'device_has_android',
    'device_has_ios',
    'device_is_multi_platform',
    'device_days_since_last_active',
]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes_flat = axes.flatten()

for ax, feat in zip(axes_flat, key_features):
    if feat not in features.columns:
        ax.axis('off')
        continue
    data = features[feat]
    is_flag = feat.startswith('device_has_') or feat == 'device_is_multi_platform'
    is_days = feat == 'device_days_since_last_active'

    if is_flag or data.nunique() <= 2:
        vc = data.value_counts().sort_index()
        ax.bar([str(v) for v in vc.index], vc.values, color='steelblue', alpha=0.75)
        ax.set_title(f'{feat}')
        ax.set_ylabel('count')
    elif is_days:
        # Clip a 365 para ver distribución "real" (sentinel 9999 se colapsa)
        clipped = data.clip(0, 365)
        ax.hist(clipped, bins=60, color='tomato', alpha=0.7)
        ax.set_title(f'{feat} (clipped 0-365)')
    else:
        positive = data[data > 0]
        if len(positive) > 0:
            ax.hist(np.log1p(positive), bins=50, color='steelblue', alpha=0.7)
            ax.set_title(f'{feat} (log1p, n={len(positive):,})')
        else:
            ax.set_title(f'{feat} (sin datos)')

# Extra: primary_platform como barra
if len(axes_flat) > len(key_features):
    ax = axes_flat[len(key_features)]
    vc = features['device_primary_platform'].value_counts()
    ax.bar(vc.index.astype(str), vc.values, color='goldenrod', alpha=0.75)
    ax.set_title('device_primary_platform')
    ax.set_ylabel('count')
    ax.tick_params(axis='x', rotation=30)

# Apagar ejes sobrantes
for ax in axes_flat[len(key_features) + 1:]:
    ax.axis('off')

plt.tight_layout()
plt.savefig(REPORT_DIR / 'features_distributions.png', dpi=100, bbox_inches='tight')
plt.close()
log_step('ANALYSIS', 'Histogramas guardados en features_distributions.png')

[ANALYSIS] 13:13:28 — Histogramas guardados en features_distributions.png


In [17]:
# [EXEC] Guardar features_devices_cutoff.parquet
features.to_parquet(PARQUET_FEATURES, index=False, compression='snappy')
size_mb = PARQUET_FEATURES.stat().st_size / 1e6
print(f"Guardado: {PARQUET_FEATURES} ({size_mb:.1f} MB)")
log_step('EXEC', f'features_devices_cutoff.parquet: {features.shape}, {size_mb:.1f} MB')

Guardado: /Users/jezquerro/Documents/tfg/data/data_qc/features_devices_cutoff.parquet (1.1 MB)
[EXEC] 13:13:28 — features_devices_cutoff.parquet: (25200, 11), 1.1 MB


In [18]:
# [EXEC] Guardar muestra y liberar memoria
features.head(20).to_csv(REPORT_DIR / 'sample_head.csv', index=False)
log_step('EXEC', 'sample_head.csv guardado')

del df
gc.collect()
print("Memoria liberada")

[EXEC] 13:13:28 — sample_head.csv guardado
Memoria liberada


## 6. Informe de ejecución

In [19]:
# [REPORT] Generar execution_report.md
t_total = time.time() - NOTEBOOK_START
t_min = int(t_total // 60)
t_sec = int(t_total % 60)
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

n_con_dev = int((features['device_records_total'] > 0).sum())
n_sin_dev = len(features) - n_con_dev

lines = []
lines += [
    "# Informe de ejecucion — devices.csv",
    "",
    f"**Notebook:** notebooks/fase1_cleaning/02g_devices.ipynb",
    f"**Fecha:** {now_str}",
    f"**Tiempo de ejecucion:** {t_min} min {t_sec}s",
    f"**CSV de entrada:** data/data_raw/devices.csv (144 MB, {n_before:,} filas, 6 cols)",
    f"**Parquet de salida:** data/data_qc/features_devices_cutoff.parquet ({features.shape[0]:,} × {features.shape[1]} cols)",
    "",
    "---", "",
    "## Resumen ejecutivo",
    "",
    f"Se procesaron {n_before:,} registros de devices. Tras filtros point-in-time",
    f"(created_at y updated_at <= REFERENCE_DATE) y filtro por sample, quedaron",
    f"{n_in_sample:,} registros ({n_in_sample/n_before*100:.1f}%). Se generaron",
    f"{features.shape[1] - 1} features en 3 grupos (volumen, plataforma, temporales).",
    "",
    f"- Usuarios con devices: {n_con_dev:,} ({n_con_dev/len(features)*100:.2f}%)",
    f"- Usuarios sin devices: {n_sin_dev:,} ({n_sin_dev/len(features)*100:.2f}%)",
    "",
    "---", "",
    "## Política point-in-time",
    "",
    f"CUTOFF = REFERENCE_DATE = {REFERENCE_DATE} (unix: {CUTOFF_DATE_UNIX}).",
    "",
    "Se descartan filas con:",
    f"- `created_at > CUTOFF_DATE_UNIX` ({n_no_nan - n_created_ok:,} filas).",
    f"- `updated_at > CUTOFF_DATE_UNIX` ({n_created_ok - n_updated_ok:,} filas).",
    "",
    "Rationale: si el registro del device no existía (o no tenía esa actividad)",
    "en el cutoff, no debe contribuir a features que describen el estado del",
    "usuario en ese momento.",
    "",
    "---", "",
    "## Constantes utilizadas",
    "",
    "| Constante | Valor |",
    "|---|---|",
    f"| `REFERENCE_DATE` | {REFERENCE_DATE} |",
    f"| `CUTOFF_DATE_UNIX` | {CUTOFF_DATE_UNIX} |",
    f"| `N_SAMPLE` | {N_SAMPLE:,} |",
    f"| `SENTINEL_DAYS` | {SENTINEL_DAYS} (nunca) |",
    "",
    "---", "",
    "## Pasos ejecutados", "",
]
for s in steps_log:
    lines.append(f"- {s}")
# ── Fix post-ejecución: parser de timestamps ──
lines += [
    "## Fix post-ejecución: parser de timestamps",
    "",
    "### Problema detectado",
    "",
    "Tras la primera ejecución, las features temporales mostraron valores",
    "imposibles:",
    "",
    "- `device_first_seen_dt` = 1970-01-21",
    "- `device_days_since_last_active` = 20527 (~56 años)",
    "",
    "Adicionalmente, los filtros point-in-time eliminaron 0 filas.",
    "",
    "### Causa raíz",
    "",
    "En pandas 3.x (y opcionalmente desde 2.0), `pd.to_datetime(..., utc=True)`",
    "devuelve un dtype `datetime64[us, UTC]` (microsegundos), no `[ns]` como",
    "en versiones antiguas. Por tanto:",
    "",
    "```",
    ".astype('int64') // 10**9",
    "```",
    "",
    "produce el número de microsegundos dividido por 1.000.000.000, lo que",
    "equivale a SEGUNDOS DIVIDIDOS POR 1000. Los valores resultantes son",
    "~1.000 veces menores de lo esperado y, al reinterpretarse como unix",
    "seconds, sitúan todas las fechas en enero de 1970.",
    "",
    "El assert añadido (`min > 1e9`) disparó correctamente:",
    "",
    "```",
    "AssertionError: created_at_unix corrupto: min=1637153.",
    "Debería ser ~1.5e9-1.8e9 (años 2017-2026).",
    "```",
    "",
    "El valor 1637153 corresponde a 2021-11-17 en microsegundos // 10**9 (truncado).",
    "",
    "### Solución aplicada",
    "",
    "Conversión robusta basada en resta de epoch con `pd.Timedelta`:",
    "",
    "```python",
    "EPOCH_UTC = pd.Timestamp('1970-01-01', tz='UTC')",
    "unix = ((dt - EPOCH_UTC) // pd.Timedelta('1s')).astype('Int64')",
    "```",
    "",
    "Esta forma es **independiente de la unidad interna de pandas** (ns/us/ms/s)",
    "y maneja NaT correctamente con dtype `Int64` nullable (sin necesidad de",
    "sanear INT64_MIN manualmente).",
    "",
    "### Validación post-fix",
    "",
    "- Assert de rango unix (1e9 < min, max < 2e9): pasa",
    "- Fechas en rango ~2018-2026 (correcto)",
    "- `device_days_since_last_active` < 365 para mayoría de usuarios activos",
    "- Filtros point-in-time eliminan algunas filas (>0)",
    "",
    "---", "",
]
lines += [
    "## Fix post-ejecución: parser de timestamps",
    "",
    "### Problema detectado",
    "",
    "Tras la primera ejecución, las features temporales mostraron valores",
    "imposibles:",
    "",
    "- `device_first_seen_dt` = 1970-01-21",
    "- `device_days_since_last_active` = 20527 (~56 años)",
    "",
    "Adicionalmente, los filtros point-in-time eliminaron 0 filas, cuando",
    "deberían haber eliminado al menos algunas (hay devices con `updated_at`",
    "posterior a REFERENCE_DATE en el CSV).",
    "",
    "### Causa",
    "",
    "Error en la conversión de timestamps ISO8601 a unix seconds. La unidad",
    "de medida se confundió en algún paso del parser, dejando los timestamps",
    "en una escala incorrecta.",
    "",
    "### Solución aplicada",
    "",
    "- Parser explícito: `pd.to_datetime(utc=True).astype('int64') // 10**9`",
    "- Asserts de validación de rango antes y después de la conversión:",
    "  - `unix > 1e9 AND unix < 2e9` (años 2001-2033)",
    "  - `year >= 2018 AND year <= 2027` post-conversión a datetime",
    "- Reconstrucción explícita de datetimes con `pd.to_datetime(unix, unit='s')`",
    "  (el parámetro `unit='s'` es CRÍTICO: sin él sale 1970)",
    "",
    "### Resultado tras fix",
    "",
    "- Fechas en rango 2018-2026 (correcto)",
    "- `device_days_since_last_active` < 365 días para usuarios activos",
    "- Filtros point-in-time aplicados correctamente",
    "",
    "---", "",
]
lines += [
    "",
    "---", "",
    "## Filtrado aplicado",
    "",
    "| Paso | Filas | % original |",
    "|---|---|---|",
    f"| CSV original | {n_before:,} | 100.00% |",
    f"| Tras eliminar user_id NaN | {n_no_nan:,} | {n_no_nan/n_before*100:.2f}% |",
    f"| Tras created_at <= REF | {n_created_ok:,} | {n_created_ok/n_before*100:.2f}% |",
    f"| Tras updated_at <= REF | {n_updated_ok:,} | {n_updated_ok/n_before*100:.2f}% |",
    f"| Tras filtro sample | {n_in_sample:,} | {n_in_sample/n_before*100:.2f}% |",
    "",
    "---", "",
    "## Features generadas (10 + user_id)",
    "",
    "### Grupo A — Volumen de devices (3)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `device_records_total` | N registros en devices.csv (reinstalaciones/sesiones) |",
    "| `device_unique_models` | N modelos distintos de hardware |",
    "| `device_platform_count` | N plataformas distintas |",
    "",
    "### Grupo B — Plataforma (4)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `device_has_android` | 1 si ha usado alguna vez android |",
    "| `device_has_ios` | 1 si ha usado alguna vez ios |",
    "| `device_is_multi_platform` | 1 si usó ambas android+ios |",
    "| `device_primary_platform` | Plataforma con más registros ('none' si no tiene) |",
    "",
    "### Grupo C — Temporales point-in-time (3)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `device_first_seen_dt` | min(created_at) (NaT si no tiene) |",
    "| `device_last_active_dt` | max(updated_at) (NaT si no tiene) |",
    "| `device_days_since_last_active` | días hasta REFERENCE_DATE (9999 si nunca) |",
    "",
    "---", "",
    "## Sanity checks aplicados (9)",
    "",
    "- [x] Shape = (N_SAMPLE, 11)",
    "- [x] user_id único",
    "- [x] Counts no-negativos",
    "- [x] `records>0 → platform_count >= 1`",
    "- [x] `is_multi_platform=1 → has_android=1 AND has_ios=1`",
    "- [x] `is_multi_platform=1 → platform_count >= 2`",
    "- [x] Al menos un usuario con android o ios (cobertura razonable)",
    "- [x] `first_seen_dt <= last_active_dt` (cuando ambos no-NaT)",
    "- [x] `days_since==9999 ↔ last_active_dt is NaT`",
    "",
    "---", "",
    "## Lo que ha ido bien",
    "",
    "- Filtros point-in-time aplicados de forma consistente con 02f",
    "- ~9 devices/user de media confirmado como patrón del dataset",
    "- device_records_total y device_unique_models son features separadas y complementarias",
    "- Sentinel 9999 permite distinguir sin ambigüedad 'nunca activo'",
    "",
    "---", "",
    "## Lo que ha ido mal o requirió ajustes",
    "",
    f"- Ratio ~{n_before / max(1, n_users_with_devices):.1f} registros/usuario en el CSV crudo:",
    "  muy alto para un F2P móvil. Probablemente cada actualización del device",
    "  crea una fila (log de sesiones). No rompe nada pero impacta la interpretación",
    "  de `device_records_total`.",
    "",
    "---", "",
    "## Decisiones tomadas",
    "",
    "- Política estricta point-in-time: descartar filas con created_at>REF O updated_at>REF",
    "- Sentinel 9999 para `device_days_since_last_active` cuando no hay devices",
    "- 'none' como categoría para `device_primary_platform` cuando no hay devices",
    "- NaT en datetimes cuando no hay devices (no se rellena)",
    "- Desempate alfabético de plataforma primaria (android < ios < otras)",
    "",
    "---", "",
    "## Archivos generados",
    "",
    "| Archivo | Descripción |",
    "|---|---|",
    "| features_devices_cutoff.parquet (11 cols) | Parquet final |",
    "| nulos_por_columna_raw.csv | Nulos CSV crudo |",
    "| nulos_features_final.csv | Nulos features finales |",
    "| features_describe.csv | Estadísticas descriptivas |",
    "| features_distributions.png | Histogramas features clave |",
    "| sample_head.csv | 20 primeras filas del parquet |",
    "| execution_report.md | Este informe |",
    "",
    "---", "",
    "## Advertencias para notebooks siguientes",
    "",
    f"- REFERENCE_DATE = {REFERENCE_DATE}",
    "- `device_primary_platform` es categórica: 1-hot en 02z.",
    "- `device_days_since_last_active == 9999` identifica usuarios sin devices.",
    "- `device_records_total` NO mide dispositivos físicos; mide logs/sesiones.",
    "",
    "---", "",
    "## Tareas pendientes",
    "",
    "- [ ] Cross-check con users.created_at: ¿divergen device_first_seen_dt y",
    "      el registro del usuario? (usuarios registrados sin device, devices sin user)",
    "- [ ] En 02z: 1-hot encoding de device_primary_platform",
    "- [ ] Investigar por qué ~9 devices/user de media (reinstalaciones? sesiones?",
    "      ¿el sistema loguea cada cambio de modelo o cada actualización?)",
    "- [ ] Analizar correlación device_days_since_last_active vs churn en EDA",
    "- [ ] Considerar feature derivada `device_lifespan_days = last_active - first_seen`",
    "",
    "---", "",
    "## Diagrama del pipeline",
    "",
    "```",
    f"devices.csv ({n_before:,} filas, 6 cols)",
    "    │",
    "    ├─ Parse ISO8601 → unix seconds (created_at, updated_at)",
    f"    ├─ Eliminar user_id NaN            (-{n_before - n_no_nan:,})",
    f"    ├─ Filtrar created_at <= REF        (-{n_no_nan - n_created_ok:,})",
    f"    ├─ Filtrar updated_at <= REF        (-{n_created_ok - n_updated_ok:,})",
    f"    └─ Filtrar por sample_user_ids      (-{n_updated_ok - n_in_sample:,})",
    "",
    f"Filtrado ({n_in_sample:,} filas)",
    "    │",
    "    ├─ Grupo A: volumen (3)",
    "    ├─ Grupo B: plataforma (4)",
    "    ├─ Grupo C: temporales point-in-time (3)",
    "    └─ Reindex con sample_user_ids + fillna",
    "",
    f"features_devices_cutoff.parquet ({features.shape[0]:,} × {features.shape[1]})",
    "```",
    "",
    "---", "",
    "## Reproducibilidad",
    "",
    "1. Ejecutar 02a_users.ipynb primero (genera sample_user_ids)",
    "2. Ejecutar 02g_devices.ipynb",
    "3. Verificar: features_devices_cutoff.parquet == 126.253 filas × 11 cols",
    "",
    "---", "",
    "## Referencias",
    "",
    "- PLANTILLA_NOTEBOOK_02.md — estructura estándar de notebook Fase 1.",
    "- 02f_iaps.ipynb — referencia para sanity checks y cutoff point-in-time.",
    "",
]

report_path = REPORT_DIR / 'execution_report.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f"Informe guardado: {report_path}")
log_step('REPORT', 'execution_report.md generado')

Informe guardado: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/devices/execution_report.md
[REPORT] 13:13:28 — execution_report.md generado


In [20]:
# [REPORT] Resumen final en consola
elapsed = time.time() - NOTEBOOK_START

print("=" * 70)
print("RESUMEN FINAL — Notebook 02g_devices")
print("=" * 70)
print(f"  Tiempo total          : {int(elapsed//60)}m {int(elapsed%60)}s")
print(f"  Filas CSV original    : {n_before:,}")
print(f"  Filas filtradas       : {n_in_sample:,} ({n_in_sample/n_before*100:.1f}%)")
print(f"  Usuarios con devices  : {int((features['device_records_total']>0).sum()):,}")
print(f"  Usuarios sin devices  : {int((features['device_records_total']==0).sum()):,}")
print(f"  has_android           : {int(features['device_has_android'].sum()):,}")
print(f"  has_ios               : {int(features['device_has_ios'].sum()):,}")
print(f"  is_multi_platform     : {int(features['device_is_multi_platform'].sum()):,}")
print(f"  Filas features final  : {len(features):,} (== {N_SAMPLE:,} sample)")
print(f"  Columnas features     : {features.shape[1]}")
print()
print("device_primary_platform value_counts:")
print(features['device_primary_platform'].value_counts().to_string())
print()
print("Archivos generados:")
print(f"  features_devices_cutoff.parquet ({PARQUET_FEATURES.stat().st_size/1e6:.1f} MB)")
print(f"  execution_report.md")
print(f"  Gráficos y CSVs en {REPORT_DIR}")
print("=" * 70)
print("Siguiente paso: revisar execution_report.md y compartirlo con Claude")

RESUMEN FINAL — Notebook 02g_devices
  Tiempo total          : 0m 5s
  Filas CSV original    : 1,129,784
  Filas filtradas       : 23,534 (2.1%)
  Usuarios con devices  : 20,708
  Usuarios sin devices  : 4,492
  has_android           : 17,101
  has_ios               : 3,625
  is_multi_platform     : 18
  Filas features final  : 25,200 (== 25,200 sample)
  Columnas features     : 11

device_primary_platform value_counts:
device_primary_platform
android    17095
none        4492
ios         3613

Archivos generados:
  features_devices_cutoff.parquet (1.1 MB)
  execution_report.md
  Gráficos y CSVs en /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/devices
Siguiente paso: revisar execution_report.md y compartirlo con Claude


In [21]:
# [REPORT] Logging dual: HTML + sección enriquecida del report
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from notebook_logging_template import (
    export_notebook_to_html, build_enriched_report_section,
)

notebook_path = PROJECT_ROOT / 'notebooks' / 'fase1_cleaning' / '02g_devices.ipynb'
html_path = REPORT_DIR / '02g_devices_run.html'
export_notebook_to_html(notebook_path, html_path)

enriched = build_enriched_report_section(
    df_final=features,
    raw_shape=(n_before, 6),
    filter_steps=[
        ('CSV original', n_before),
        ('Sin user_id NaN', n_no_nan),
        ('created_at <= REF', n_created_ok),
        ('updated_at <= REF', n_updated_ok),
        ('En sample', n_in_sample),
    ],
)
with open(REPORT_DIR / 'execution_report.md', 'a', encoding='utf-8') as f:
    f.write('\n\n---\n\n' + enriched)
print(f"Enriquecimiento appendeado a {REPORT_DIR / 'execution_report.md'}")

HTML generado: 02g_devices_run.html (0.5 MB)
Enriquecimiento appendeado a /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/devices/execution_report.md
